## Imports

In [ ]:
from __future__ import absolute_import, division, print_function, unicode_literals
import tensorflow as tf


import sys
# !{sys.executable} -m pip install git+https://github.com/nottombrown/imagenet_stubs
# sys.path.append("..")

# %matplotlib inline

tf.compat.v1.disable_eager_execution()

# import imagenet_stubs
import numpy as np
import tensorflow.keras
from tensorflow.keras.preprocessing import image
# from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input, decode_predictions
# from tensorflow.keras.layers import Dense, Flatten
# from tensorflow.keras.models import Model
# import tensorflow.keras.backend as k
from matplotlib import pyplot as plt
from IPython.display import clear_output, Image, display
import cv2

# from art.estimators.classification import KerasClassifier
from art.attacks.evasion import HopSkipJump
from art.utils import to_categorical

from PIL import Image
import matplotlib.image as mpimg

import shutil
import os
import subprocess

## Functions

In [ ]:
import os
import math
import numpy as np
from PIL import Image, ExifTags

def load_and_preprocess_image(image_path):
    img = image.load_img(image_path)
    img = image.img_to_array(img)
    return img

In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np

def load_and_preprocess_image(image_path):
    img = image.load_img(image_path)  # Load the image
    img = image.img_to_array(img)  # Convert the image to a numpy array
    return img

In [ ]:
import subprocess
import time
import os

def run_DLoopDetector(data_number):
    directory = f'DLoopDetector_demo/sequence_{data_number}'
    command = f'./demo_brief > DLoopDetector_demo/sequence_{data_number}/results/noisy/output.txt'

    try:
        os.chdir(directory)
        # Start the command and immediately attempt to close any new windows it opens
        process = subprocess.Popen(command, shell=True)

        # Wait a bit for the application to open its windows
        time.sleep(10)  # Adjust as necessary

        image_variable = manage_windows("Trajectory", "Trajectory", "Current image")

        process.wait()  # Wait for the process to complete

        return image_variable
        
    except Exception as e:
        # print(f"An error occurred: {e}")
        return None

In [ ]:
def extract_image_names_with_loops(filename):
    # List to store the image names that match the condition
    image_names = []
    # Open the file and read the lines
    with open(filename, 'r') as file:
        lines = file.readlines()
    
    # Iterate through lines, checking each pair
    for i in range(len(lines) - 1):  # Ensure there is a next line to check
        # Check current and next line for the specified pattern
        if "Adding image" in lines[i] and "Loop found with image" in lines[i+1]:
            # Extract the image name from the current line
            # The image name starts after the last colon ':' and before the file extension '.png'
            start_index = lines[i].rfind(':') + 2
            end_index = lines[i].rfind('.png') + 4  # Include '.png' in the extracted name
            full_image_path = lines[i][start_index:end_index]
            # Extract just the image name after the last slash '/'
            image_name = full_image_path.split('/')[-1]
            # Add the extracted image name to the list
            image_names.append(image_name)

    return image_names

In [ ]:
def check_no_loop_for_image(filename, image_name_to_check):
    # Open the file and read the lines
    with open(filename, 'r') as file:
        lines = file.readlines()
    
    # Iterate through lines to find the image
    for i in range(len(lines) - 1):  # Ensure there is a next line to check
        # Check current line for "Adding image" and the specific image name
        if "Adding image" in lines[i] and image_name_to_check in lines[i]:
            # Check if the next line indicates a loop was found
            if "Loop found with image" in lines[i + 1]:
                return False  # Loop was found for this image
            else:
                return True  # Loop was not found for this image

    # If the image name was not found in the file at all, you might want to handle it differently
    return False

In [ ]:
def delete_image(img):
    os.remove(img)

In [ ]:
def initial_image_generator(path, len, target_size):
    init_images = []
    for x in range(len):
        init_image_path = path
        init_image = image.load_img(init_image_path, target_size=target_size)
        init_image = image.img_to_array(init_image)
        init_images.append(init_image)
    return np.array(init_images)

In [ ]:
def save_multi_images(images, names, directory):
    for img, name in zip(images, names):
        
        # Construct the full path where the image will be saved
        file_path = os.path.join(directory, name)
        cv2.imwrite(file_path, img)

In [ ]:
# Function to convert NumPy array to Pillow Image and save it
def save_image(image_array, save_path):
    # Convert the image from BGR (OpenCV format) to RGB
    image_rgb = cv2.cvtColor(image_array, cv2.COLOR_BGR2RGB)
    # Convert the NumPy array to a Pillow Image
    pil_image = Image.fromarray(image_rgb)
    # Save the image using Pillow
    pil_image.save(save_path)

In [ ]:
def delete_images(names, directory):
    """
    Deletes a list of images from the specified directory.

    Args:
    names (list of str): The list of image filenames to delete.
    directory (str): The directory from which to delete the images.
    """
    for name in names:
        # Construct the full path to the image
        file_path = os.path.join(directory, name)
        
        # Check if the file exists before attempting to delete it
        if os.path.exists(file_path):
            os.remove(file_path)
            # print(f"Deleted {file_path}")

In [ ]:
import subprocess
import pyautogui
from PIL import ImageGrab, Image
import time

def find_window(title):
    try:
        output = subprocess.check_output(['wmctrl', '-lG'], encoding='utf-8')
        for line in output.splitlines():
            if title in line:
                # print(f"Window found: {line}")
                return line
        # print(f"No window with title '{title}' found.")
        return None
    except subprocess.CalledProcessError as e:
        # print(f"Error finding window: {e}")
        return None

def take_screenshot_of_window(window_line):
    try:
        data = window_line.split()
        x, y, width, height = map(int, data[2:6])

        # Adjust the y-coordinate to move the window half an inch upwards
        y_adjusted = y - 48

        # Move the window using xdotool
        window_id = data[0]
        subprocess.run(['xdotool', 'windowmove', window_id, str(x), str(y_adjusted)], check=True)
        
        # Give a moment for the window to move
        time.sleep(1)

        # Use PIL to take a screenshot
        bbox = (x, y_adjusted, x + width, y_adjusted + height)
        screenshot = ImageGrab.grab(bbox)
        return screenshot
    except Exception as e:
        # print(f"Error taking screenshot: {e}")
        return None

def close_window(window_id):
    try:
        subprocess.run(['wmctrl', '-ic', window_id], check=True)
        # print(f"Closed window with ID: {window_id}")
    except subprocess.CalledProcessError as e:
        # print(f"Error closing window: {e}")
        return None

def manage_windows(title_to_capture, *titles_to_close):
    
    window_line = find_window(title_to_capture)
    if not window_line:
        # print(f"No window with title '{title_to_capture}' found.")
        return None

    image = take_screenshot_of_window(window_line)
    if image is None:
        # print("Failed to take a screenshot.")
        return None

    # Close specified windows
    for title in titles_to_close:
        while True:
            window_line = find_window(title)
            if not window_line:
                break
            window_id = window_line.split()[0]
            close_window(window_id)
            time.sleep(1)  # Add a short delay to ensure the window is closed properly

    return image

In [ ]:
import numpy as np
from skimage.metrics import structural_similarity as ssim
import cv2

def calculate_ssim(image1, image2):
    # Load images if paths are provided
    if isinstance(image1, str):
        image1 = cv2.imread(image1)
    if isinstance(image2, str):
        image2 = cv2.imread(image2)
    
    # Convert images to grayscale if they are RGB
    if len(image1.shape) > 2:
        image1 = cv2.cvtColor(image1, cv2.COLOR_BGR2GRAY)
    if len(image2.shape) > 2:
        image2 = cv2.cvtColor(image2, cv2.COLOR_BGR2GRAY)
    
    # Calculate the range of pixel values
    data_range = np.max(image1) - np.min(image1)
    
    # Calculate SSIM between the two images
    ssim_value, _ = ssim(image1, image2, full=True, data_range=data_range)
    
    return ssim_value*100

## Attacks

In [ ]:
from art.estimators.classification import BlackBoxClassifier
from art.attacks.evasion import HopSkipJump
from datetime import datetime
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
import cv2
import os
import shutil
import math

import warnings

# Suppress all warnings
warnings.filterwarnings("ignore")

sequences=[1,2,3,4,5,6,7]

def predict(x):
    outcomes=[]
    
    delete_images([image_name], f"DLoopDetector_demo/sequence_{sequence}/resources/images")
    cv2.imwrite(f"DLoopDetector_demo/sequence_{sequence}/resources/images/{image_name}", cv2.cvtColor(x[0],cv2.COLOR_BGR2RGB))
    image_variable = run_DLoopDetector(sequence)

    if(check_no_loop_for_image(f"DLoopDetector_demo/sequence_{sequence}/results/noisy/output.txt", image_name)):
        if image_variable:
            image_variable.save(f"DLoopDetector_demo/sequence_{sequence}/results/noisy/trajectory.png")
        cv2.imwrite(f"DLoopDetector_demo/sequence_{sequence}/results/noisy/{image_name}", cv2.cvtColor(x[0],cv2.COLOR_BGR2RGB))
        outcomes.append(1)
    else:
        outcomes.append(0)
    print("Outcomes: ", outcomes)
    return to_categorical(outcomes, 2)

with tf.device('/device:GPU:0'): 
    
    iter_step = 10
    
    start_time_total = datetime.now()

    # Loop over sequences
    for sequence in sequences:
        
        start_time_sequence = datetime.now()
        
        print("*"*20)
        print(f"Attacking Sequence {sequence}")
        print("*"*20)
    
        images = extract_image_names_with_loops(f"DLoopDetector_demo/sequence_{sequence}/results/original/output.txt")
        print(f"Loops found in sequence {sequence}: {len(images)}")

        mean_ssim = []

        # Loop over images
        for image_name in images:
            start_time_image = datetime.now()

            original_image = load_and_preprocess_image(f"DLoopDetector_demo/sequence_{sequence}/resources/images/{image_name}")
            original_image = original_image.astype(np.float32)
            original_image = np.expand_dims(original_image, axis=0)
            
            image_for_noise = load_and_preprocess_image("path_to_image_for_noise")  # Replace with the actual path to the image you want to use for noise
            image_for_noise = image_for_noise.astype(np.float32)
            image_for_noise = cv2.resize(image_for_noise, (original_image[0].shape[1], original_image[0].shape[0]))
            image_for_noise = np.expand_dims(image_for_noise, axis=0)
            
            classifier = BlackBoxClassifier(predict, image_for_noise[0].shape, 2, clip_values=(0, 255))
            attack = HopSkipJump(classifier=classifier, targeted=True, norm=2, max_iter=0, max_eval=2, init_eval=1)

            # HSJA outer loops
            for t in range(2):
                print("*"*20)
                print(f"Image {image_name} at iteration {t+1}")
                # Generate adversarial examples for the batch
                x_adv = attack.generate(x=original_image, y=to_categorical([1], 2), x_adv_init=image_for_noise, resume=True)
    
                attack.max_iter = iter_step

                print("*"*20)
                
            plt.imshow(x_adv[0].astype(np.uint8))
            plt.axis('off')  # Hide axis
            plt.show()

            print("*"*20)
            print(f"Time taken for iteration on image {image_name}: {datetime.now() - start_time_image}")
            ssim1 = calculate_ssim(f"/home/romi/Desktop/mmhashir/city_center_png/{image_name}", x_adv[0])
            print(f"SSIM of image {image_name}: {ssim1}")
            mean_ssim.append(ssim1)
            print("*"*20)

        print("*"*20)
        print(f"Loop Closures in Sequence {sequence} before attack: {len(images)}")
        lcaa = len(extract_image_names_with_loops(f"DLoopDetector_demo/sequence_{sequence}/results/noisy/output.txt"))
        print(f"Loop Closures in Sequence {sequence} after attack: {lcaa}")
        print(f"Average SSIM of Attacked Images in Sequence {sequence}: {sum(mean_ssim)/len(mean_ssim)}")
        print(f"Time Taken on Sequence {sequence}: {datetime.now()-start_time_sequence}")
        print("*"*20)
        
        image1 = Image.open(f"DLoopDetector_demo/sequence_{sequence}/results/original/trajectory.png")
        image2 = Image.open(f"DLoopDetector_demo/sequence_{sequence}/results/noisy/trajectory.png")
        
        # Display the images in a row with headings
        fig, ax = plt.subplots(1, 2, figsize=(10, 5))
        
        # Display the first image with heading
        ax[0].imshow(image1)
        ax[0].set_title('Original Trajectory')
        ax[0].axis('off')  # Hide the axes
        
        # Display the second image with heading
        ax[1].imshow(image2)
        ax[1].set_title('Trajectory After Attack')
        ax[1].axis('off')  # Hide the axes
        
        plt.show()